# $k_{\max}$ ablation for the FHN-FNO surrogate

Sweeps $k_{\max} \in \{4, 8, 16, 32, 64\}$ with 3 seeds each on the 8000-sample
FHN dataset, using the same FiLM-conditioned FNO as the paper (width 64, 6 Fourier
layers). Reports per-sample relative $L^2$ error on $u$ and $v$.

Put `fhn_1d_8000.h5` in `MyDrive/fhn_kmax_ablation/`; results land in the same folder.

Runtime: A100 GPU. The sweep is resumable, so a disconnect doesn't waste finished runs.

## Check GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
import torch
print('CUDA available:', torch.cuda.is_available(), '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

No A100 (or at least a V100/L4)? Runtime > Change runtime type > GPU, then re-run.

## Install deps and mount Drive

In [ ]:
!pip install -q h5py
from google.colab import drive
drive.mount('/content/drive')

## Locate the dataset

Edit `DRIVE_FOLDER` and `DATASET_NAME` if yours are different.

In [ ]:
import os, shutil

DRIVE_FOLDER = '/content/drive/MyDrive/fhn_kmax_ablation'
DATASET_NAME = 'fhn_1d_8000.h5'

DRIVE_DATA = os.path.join(DRIVE_FOLDER, DATASET_NAME)
LOCAL_DATA = f'/content/{DATASET_NAME}'

assert os.path.exists(DRIVE_FOLDER), (
    f'Folder not found: {DRIVE_FOLDER}. Create it in your Drive and place '
    f'{DATASET_NAME} inside.')
assert os.path.exists(DRIVE_DATA), (
    f'Dataset not found at {DRIVE_DATA}. Upload {DATASET_NAME} to that folder.')

if not os.path.exists(LOCAL_DATA):
    print(f'Copying {DATASET_NAME} from Drive to local disk for fast I/O...')
    shutil.copy(DRIVE_DATA, LOCAL_DATA)
print(f'Dataset ready: {LOCAL_DATA}  ({os.path.getsize(LOCAL_DATA)/1e9:.2f} GB)')

# results go back to Drive
DRIVE_OUT = DRIVE_FOLDER
print(f'Results will be written to: {DRIVE_OUT}')

## FNO model and DataConfig

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DataConfig:
    """FHN parameter ranges, must match the dataset."""
    Du_range  = (0.01, 0.1)
    Dv_range  = (0.005, 0.05)
    a_range   = (-0.1, 0.1)
    b_range   = (0.1, 0.5)
    tau_range = (1.0, 20.0)

class SpectralConv1d(nn.Module):
    def __init__(self, in_c, out_c, modes):
        super().__init__()
        self.in_c, self.out_c, self.modes = in_c, out_c, modes
        scale = 1 / (in_c * out_c)
        self.weights = nn.Parameter(scale * torch.randn(in_c, out_c, modes, dtype=torch.cfloat))
    def forward(self, x):
        B, _, nx = x.shape
        xf = torch.fft.rfft(x, dim=-1)
        out = torch.zeros(B, self.out_c, nx // 2 + 1, dtype=torch.cfloat, device=x.device)
        out[:, :, :self.modes] = torch.einsum('bix,iox->box', xf[:, :, :self.modes], self.weights)
        return torch.fft.irfft(out, n=nx, dim=-1)

class FourierLayer(nn.Module):
    def __init__(self, width, modes):
        super().__init__()
        self.spectral = SpectralConv1d(width, width, modes)
        self.w = nn.Conv1d(width, width, 1)
        self.norm = nn.InstanceNorm1d(width, affine=True)
        self.residual_weight = nn.Parameter(torch.ones(1))
    def forward(self, x):
        residual = x
        out = F.gelu(self.spectral(x) + self.w(x))
        out = self.norm(out)
        return out + self.residual_weight * residual

class ParameterEncoder(nn.Module):
    def __init__(self, n_params, width):
        super().__init__()
        h = width * 2
        self.encoder = nn.Sequential(
            nn.Linear(n_params, h), nn.GELU(), nn.LayerNorm(h),
            nn.Linear(h, width), nn.GELU(),
            nn.Linear(width, width), nn.LayerNorm(width))
    def forward(self, p):
        return self.encoder(p)

class FNO(nn.Module):
    """FiLM-conditioned 1D FNO, same as the paper."""
    def __init__(self, modes=16, width=64, n_layers=6, in_channels=2, out_channels=2,
                 use_param_conditioning=True, n_params=5):
        super().__init__()
        self.modes, self.width, self.n_layers = modes, width, n_layers
        self.use_param_conditioning = use_param_conditioning
        if use_param_conditioning:
            self.param_encoder = ParameterEncoder(n_params, width)
            self.param_film_gamma = nn.ModuleList([nn.Linear(width, width) for _ in range(n_layers)])
            self.param_film_beta  = nn.ModuleList([nn.Linear(width, width) for _ in range(n_layers)])
        self.lift = nn.Sequential(
            nn.Conv1d(in_channels, width * 2, 1), nn.GELU(),
            nn.Conv1d(width * 2, width, 1))
        self.projection = nn.Sequential(
            nn.Conv1d(width, width, 1), nn.GELU(),
            nn.Conv1d(width, out_channels, 1))
        self.fourier_layers = nn.ModuleList([FourierLayer(width, modes) for _ in range(n_layers)])
        self.global_residual = nn.Parameter(torch.ones(1) * 0.1)
    def forward(self, x, params=None):
        input_x = x.clone()
        x = self.lift(x)
        if self.use_param_conditioning and params is not None:
            pf = self.param_encoder(params)
        for i, layer in enumerate(self.fourier_layers):
            x = layer(x)
            if self.use_param_conditioning and params is not None:
                g = self.param_film_gamma[i](pf).unsqueeze(-1)
                b = self.param_film_beta[i](pf).unsqueeze(-1)
                x = g * x + b
        return self.projection(x) + self.global_residual * input_x

n_p = sum(p.numel() for p in FNO(modes=16).parameters())
print(f'FNO defined OK (k_max=16 reference: {n_p:,} params)')

## Dataset and per-sample relative-$L^2$ evaluator

The validation metric is per-sample relative $L^2$, averaged across validation
samples, matching the main table in the paper.

In [ ]:
import h5py, json, time
import numpy as np
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

class H5SingleStepDataset(Dataset):
    """Single-step (u,v) pairs from the HDF5, cached in RAM."""
    def __init__(self, h5_path, indices, um, us, vm, vs, pm, ps):
        with h5py.File(h5_path, 'r') as f:
            self.u = np.array(f['u_traj'][indices])
            self.v = np.array(f['v_traj'][indices])
            self.p = np.array(f['params'][indices])
        self.um, self.us, self.vm, self.vs = um, us, vm, vs
        self.pm, self.ps = pm, ps
        self.n_samples, self.n_times, self.nx = self.u.shape
    def __len__(self):
        return self.n_samples * (self.n_times - 1)
    def __getitem__(self, idx):
        i, t = divmod(idx, self.n_times - 1)
        u_in  = (self.u[i, t]   - self.um) / self.us
        v_in  = (self.v[i, t]   - self.vm) / self.vs
        u_out = (self.u[i, t+1] - self.um) / self.us
        v_out = (self.v[i, t+1] - self.vm) / self.vs
        p_norm = (self.p[i] - self.pm) / self.ps
        return (torch.from_numpy(np.stack([u_in, v_in], 0)).float(),
                torch.from_numpy(np.stack([u_out, v_out], 0)).float(),
                torch.from_numpy(p_norm).float())

def train_one(k_max, seed, train_ds, val_ds, epochs, device, lr=1e-3, bs=32, log_path=None):
    """Train one (k_max, seed) and return summary metrics."""
    torch.manual_seed(seed); np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    model = FNO(modes=k_max, width=64, n_layers=6,
                use_param_conditioning=True, n_params=5).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    tl = DataLoader(train_ds, batch_size=bs, shuffle=True, num_workers=2, pin_memory=True)
    vl = DataLoader(val_ds,   batch_size=bs, shuffle=False, num_workers=2, pin_memory=True)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    crit = nn.MSELoss()
    best_u, best_v, best_ep = float('inf'), float('inf'), -1
    with open(log_path, 'w') as flog:
        flog.write(f'# k_max={k_max} seed={seed}\n')
        for ep in range(epochs):
            model.train()
            for x, y, p in tl:
                x = x.to(device, non_blocking=True)
                y = y.to(device, non_blocking=True)
                p = p.to(device, non_blocking=True)
                opt.zero_grad()
                loss = crit(model(x, p), y)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
            # per-sample rel-L2, averaged over val
            model.eval()
            rus, rvs = [], []
            with torch.no_grad():
                for x, y, p in vl:
                    x = x.to(device); y = y.to(device); p = p.to(device)
                    pred = model(x, p)
                    nu = ((pred[:,0]-y[:,0])**2).sum(dim=-1)
                    du = (y[:,0]**2).sum(dim=-1)
                    nv = ((pred[:,1]-y[:,1])**2).sum(dim=-1)
                    dv = (y[:,1]**2).sum(dim=-1)
                    rus.append(torch.sqrt(nu / (du + 1e-12)).cpu().numpy())
                    rvs.append(torch.sqrt(nv / (dv + 1e-12)).cpu().numpy())
            rel_u = float(np.concatenate(rus).mean())
            rel_v = float(np.concatenate(rvs).mean())
            sch.step()
            flog.write(f'epoch {ep} rel_u {rel_u:.6f} rel_v {rel_v:.6f}\n')
            flog.flush()
            if rel_u + rel_v < best_u + best_v:
                best_u, best_v, best_ep = rel_u, rel_v, ep
    # forward-pass timing
    model.eval()
    xt = torch.randn(1, 2, train_ds.nx, device=device)
    pt = torch.zeros(1, 5, device=device)
    with torch.no_grad():
        for _ in range(10):
            _ = model(xt, pt)
    torch.cuda.synchronize()
    times = []
    with torch.no_grad():
        for _ in range(50):
            t0 = time.time(); _ = model(xt, pt); torch.cuda.synchronize()
            times.append(time.time() - t0)
    return {'k_max': k_max, 'seed': seed, 'n_params': n_params,
            'rel_u': best_u, 'rel_v': best_v, 'best_epoch': best_ep,
            'fwd_ms': float(np.median(times) * 1000)}

print('Ablation runner defined OK')

## Train/val splits and normalisation

In [ ]:
import numpy as np
import h5py

cfg = DataConfig()
with h5py.File(LOCAL_DATA, 'r') as f:
    N = f['u_traj'].shape[0]
    nx = f['u_traj'].shape[-1]
print(f'Dataset: {N} trajectories, nx={nx}')

rng = np.random.RandomState(42)
perm = rng.permutation(N)
n_train = int(N * 0.8)
train_idx = np.sort(perm[:n_train])
val_idx   = np.sort(perm[n_train:])
print(f'Split: {len(train_idx)} train / {len(val_idx)} val')

# stats from the training split
with h5py.File(LOCAL_DATA, 'r') as f:
    u = np.array(f['u_traj'][train_idx])
    v = np.array(f['v_traj'][train_idx])
um, us = float(u.mean()), float(u.std() + 1e-8)
vm, vs = float(v.mean()), float(v.std() + 1e-8)
del u, v
ranges = np.array([cfg.Du_range, cfg.Dv_range, cfg.a_range,
                   cfg.b_range, cfg.tau_range])
pm = ranges.mean(axis=1).astype(np.float32)
ps = ((ranges[:, 1] - ranges[:, 0]) / np.sqrt(12.0)).astype(np.float32)
print(f'u stats: mean={um:.4f} std={us:.4f}')
print(f'v stats: mean={vm:.4f} std={vs:.4f}')

train_ds = H5SingleStepDataset(LOCAL_DATA, train_idx, um, us, vm, vs, pm, ps)
val_ds   = H5SingleStepDataset(LOCAL_DATA, val_idx,   um, us, vm, vs, pm, ps)
print(f'Loaded datasets into RAM: train {len(train_ds.u)} traj, val {len(val_ds.u)} traj')

## Run the sweep

Results are written back to Drive after each run. If Colab disconnects, re-run and
the sweep skips finished runs.

In [ ]:
import os, time, json

K_VALUES = [4, 8, 16, 32, 64]
SEEDS    = [42, 1337, 2025]
EPOCHS   = 100
device   = 'cuda'

results_path = os.path.join(DRIVE_OUT, 'kmax_ablation.json')
if os.path.exists(results_path):
    with open(results_path) as f:
        all_runs = json.load(f)
    print(f'Resuming with {len(all_runs)} runs already done')
else:
    all_runs = []
done = {(r['k_max'], r['seed']) for r in all_runs}

for k in K_VALUES:
    for s in SEEDS:
        if (k, s) in done:
            print(f'== skip k={k} seed={s} (already done) ==', flush=True)
            continue
        print(f'\n== k_max={k}, seed={s} ==', flush=True)
        t0 = time.time()
        log_path = os.path.join(DRIVE_OUT, f'run_k{k}_s{s}.log')
        r = train_one(k, s, train_ds, val_ds, EPOCHS, device, log_path=log_path)
        r['wall_train_s'] = time.time() - t0
        all_runs.append(r)
        print(f'  rel_u={r["rel_u"]:.4f} rel_v={r["rel_v"]:.4f} '
              f'params={r["n_params"]:,} fwd={r["fwd_ms"]:.2f}ms '
              f'best_epoch={r["best_epoch"]} wall={r["wall_train_s"]:.0f}s', flush=True)
        with open(results_path, 'w') as f:
            json.dump(all_runs, f, indent=2)
print('\n==== ALL DONE ====')

## Summary table and figure

In [ ]:
import os, json
import numpy as np
import matplotlib.pyplot as plt

with open(results_path) as f:
    all_runs = json.load(f)

ks = sorted({r['k_max'] for r in all_runs})
summary = {}
for k in ks:
    runs = [r for r in all_runs if r['k_max'] == k]
    summary[k] = {
        'rel_u_mean': float(np.mean([r['rel_u'] for r in runs])),
        'rel_u_std':  float(np.std ([r['rel_u'] for r in runs])),
        'rel_v_mean': float(np.mean([r['rel_v'] for r in runs])),
        'rel_v_std':  float(np.std ([r['rel_v'] for r in runs])),
        'n_params':   runs[0]['n_params'],
        'fwd_ms_mean': float(np.mean([r['fwd_ms'] for r in runs])),
    }

print(f'{"k_max":>6s} | {"rel_u (mean+/-std)":>22s} | {"rel_v":>22s} | {"params":>10s} | {"fwd(ms)":>8s}')
print('=' * 78)
for k in ks:
    s = summary[k]
    print(f'{k:>6d} | {s["rel_u_mean"]:.4f} +/- {s["rel_u_std"]:.4f}    | '
          f'{s["rel_v_mean"]:.4f} +/- {s["rel_v_std"]:.4f}    | '
          f'{s["n_params"]:>10,d} | {s["fwd_ms_mean"]:>8.2f}')

with open(os.path.join(DRIVE_OUT, 'kmax_ablation_summary.json'), 'w') as f:
    json.dump({str(k): v for k, v in summary.items()}, f, indent=2)

plt.rcParams['font.family'] = 'serif'
fig, axes = plt.subplots(1, 2, figsize=(8, 3.2))
ksA = np.array(ks, dtype=float)
ru_m = np.array([summary[int(k)]['rel_u_mean'] for k in ks])
ru_s = np.array([summary[int(k)]['rel_u_std']  for k in ks])
rv_m = np.array([summary[int(k)]['rel_v_mean'] for k in ks])
rv_s = np.array([summary[int(k)]['rel_v_std']  for k in ks])
n_par = np.array([summary[int(k)]['n_params']   for k in ks]) / 1000.0

ax = axes[0]
ax.errorbar(ksA, ru_m, yerr=ru_s, marker='o', label=r'rel. $L^2$, $u$',
            color='#1f77b4', linewidth=2, capsize=3)
ax.errorbar(ksA, rv_m, yerr=rv_s, marker='s', label=r'rel. $L^2$, $v$',
            color='#ff7f0e', linewidth=2, capsize=3)
ax.axvline(16, color='gray', linestyle='--', linewidth=1, alpha=0.7,
           label=r'chosen $k_{\max}\!=\!16$')
ax.set_xscale('log', base=2)
ax.set_xticks(ks); ax.set_xticklabels([str(k) for k in ks])
ax.set_xlabel(r'$k_{\max}$')
ax.set_ylabel(r'Validation rel. $L^2$ error')
ax.set_title('Accuracy')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8, loc='best')

ax = axes[1]
ax.plot(ksA, n_par, marker='o', color='#2ca02c', linewidth=2)
ax.axvline(16, color='gray', linestyle='--', linewidth=1, alpha=0.7)
ax.set_xscale('log', base=2)
ax.set_xticks(ks); ax.set_xticklabels([str(k) for k in ks])
ax.set_xlabel(r'$k_{\max}$')
ax.set_ylabel(r'Parameter count ($\times 10^3$)')
ax.set_title('Model size')
ax.grid(True, alpha=0.3)

plt.tight_layout()
fig_path = os.path.join(DRIVE_OUT, 'kmax_ablation.png')
fig.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'\nSaved figure: {fig_path}')
print(f'Saved summary: {os.path.join(DRIVE_OUT, "kmax_ablation_summary.json")}')